# 01 — Exploración de datos

Análisis exploratorio (EDA) del dataset sintético generado por `app/use_cases/generate_dataset/`.

Acompaña al deliverable §14.3 (dataset) y §14.4 (README + demo). El notebook contesta tres preguntas:

1. ¿Qué tan equilibrado está el dataset (ramo, ciudad, tier)?
2. ¿Cada una de las 21 reglas (FS-01..14 + RF-01..07) tiene ejemplares suficientes?
3. ¿Qué tan correlacionado está el `score` agregado con la etiqueta simulada `etiqueta_fraude_simulada`?

> **Cómo correr este notebook:** `uv run jupyter notebook notebooks/01_exploracion_datos.ipynb`
> Requiere que `scripts/generate_dataset.py` haya producido los CSVs en `data/synthetic/`.


## 1. Carga del dataset

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Resolver el root del backend desde el notebook
BACKEND_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BACKEND_ROOT / "data" / "synthetic"
assert DATA_DIR.exists(), f"No existe {DATA_DIR}. Ejecuta scripts/generate_dataset.py primero."

siniestros = pd.read_csv(DATA_DIR / "siniestros.csv", parse_dates=["fecha_ocurrencia", "fecha_reporte"])
polizas = pd.read_csv(DATA_DIR / "polizas.csv", parse_dates=["fecha_inicio", "fecha_fin"])
asegurados = pd.read_csv(DATA_DIR / "asegurados.csv")
proveedores = pd.read_csv(DATA_DIR / "beneficiarios_proveedores.csv")
documentos = pd.read_csv(DATA_DIR / "documentos.csv")

# claims.json trae los siniestros ya pre-puntuados (score, tier, alertas)
with (DATA_DIR / "claims.json").open(encoding="utf-8") as f:
    claims_json = json.load(f)
claims_scored = pd.DataFrame([
    {
        "id_siniestro": c["id_siniestro"],
        "score": c["score"],
        "nivel": c["nivel"],
        "ramo": c["ramo"],
        "alertas_codigos": [a["codigo"] for a in c.get("alertas", [])],
    }
    for c in claims_json
])

print(f"siniestros:   {len(siniestros):>5} filas")
print(f"polizas:      {len(polizas):>5} filas")
print(f"asegurados:   {len(asegurados):>5} filas")
print(f"proveedores:  {len(proveedores):>5} filas")
print(f"documentos:   {len(documentos):>5} filas")
print(f"claims.json:  {len(claims_scored):>5} siniestros pre-puntuados")

## 2. Distribución del semáforo (tier)

Esperamos una distribución calibrada: ~50% verde (flujo normal), ~20% amarillo (revisión documental), ~30% rojo (revisión de campo). Calidades extremas (todo rojo o todo verde) indicarían un generador desbalanceado.

In [ ]:
tier_counts = claims_scored["nivel"].value_counts().reindex(["verde", "amarillo", "rojo"]).fillna(0)
tier_pct = (tier_counts / tier_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2ECC71", "#F1C40F", "#E74C3C"]
bars = ax.bar(tier_counts.index, tier_counts.values, color=colors)
ax.set_title("Distribución de siniestros por tier")
ax.set_ylabel("Cantidad de siniestros")
for bar, pct in zip(bars, tier_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"{int(bar.get_height())} ({pct}%)",
            ha="center", va="bottom")
plt.tight_layout()
plt.show()

display(tier_counts.to_frame("count").assign(pct=tier_pct))

## 3. Cobertura de señales (FS-01..FS-14 + RF-01..RF-07)

Cada una de las 21 reglas debe disparar en **≥ 3 siniestros** (criterio que valida el generador). Si una regla falta o tiene < 3, hay que agregar archetypes en `app/use_cases/generate_dataset/_archetypes.py`.

In [ ]:
ALL_CODES = [f"FS-{i:02d}" for i in range(1, 15)] + [f"RF-{i:02d}" for i in range(1, 8)]

fire_counter: Counter[str] = Counter()
for codes in claims_scored["alertas_codigos"]:
    fire_counter.update(codes)

coverage = pd.DataFrame(
    [(code, fire_counter.get(code, 0)) for code in ALL_CODES],
    columns=["codigo", "disparos"],
)
coverage["familia"] = coverage["codigo"].str.startswith("FS").map({True: "FS (aditiva)", False: "RF (dura)"})
coverage["cumple_minimo"] = coverage["disparos"] >= 3

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = coverage["cumple_minimo"].map({True: "#27AE60", False: "#C0392B"})
ax.bar(coverage["codigo"], coverage["disparos"], color=bar_colors)
ax.axhline(3, linestyle="--", color="black", label="mínimo (3)")
ax.set_title("Cobertura de cada regla en el dataset sintético")
ax.set_ylabel("Veces que la regla disparó")
ax.set_xlabel("Código de regla")
ax.tick_params(axis="x", labelrotation=45)
ax.legend()
plt.tight_layout()
plt.show()

faltantes = coverage[~coverage["cumple_minimo"]]
if faltantes.empty:
    print("OK: todas las 21 reglas tienen al menos 3 ejemplares.")
else:
    print("FALTAN ejemplares para:")
    display(faltantes)

display(coverage)

## 4. Score agregado vs. etiqueta de fraude simulada

`etiqueta_fraude_simulada` se deriva del tier resultante (1 si `tier == rojo`). Comparamos la distribución del `score` aditivo contra la etiqueta para validar que el motor de reglas separa claramente las dos clases — eso es la **expectativa** para que el modelo supervisado (V4) tenga una señal aprendible.

In [ ]:
merged = siniestros.merge(claims_scored[["id_siniestro", "score", "nivel"]], on="id_siniestro", how="left")

fig, ax = plt.subplots(figsize=(8, 4))
bins = list(range(0, 105, 5))
for et, color, label in [(0, "#3498DB", "etiqueta=0 (legítimo simulado)"), (1, "#E74C3C", "etiqueta=1 (posible fraude simulado)")]:
    subset = merged.loc[merged["etiqueta_fraude_simulada"] == et, "score"].dropna()
    ax.hist(subset, bins=bins, alpha=0.55, label=label, color=color)
ax.axvline(40, linestyle="--", color="#2ECC71", label="corte verde→amarillo (40)")
ax.axvline(75, linestyle="--", color="#E74C3C", label="corte amarillo→rojo (75)")
ax.set_title("Distribución del score aditivo por etiqueta")
ax.set_xlabel("Score (0-100)")
ax.set_ylabel("Cantidad de siniestros")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Composición por ramo y ciudad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ramo_counts = siniestros["ramo"].value_counts()
axes[0].barh(ramo_counts.index, ramo_counts.values, color="#34495E")
axes[0].set_title("Siniestros por ramo")
axes[0].invert_yaxis()

ciudad_counts = siniestros["sucursal"].value_counts().head(10)
axes[1].barh(ciudad_counts.index, ciudad_counts.values, color="#16A085")
axes[1].set_title("Top 10 sucursales por número de siniestros")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Distribución temporal de demoras

Demora de reporte (`dias_entre_ocurrencia_reporte`) alimenta directamente FS-12 (bandas: > 7 días → 5 pts, 4-7 días → 3 pts). Una distribución sana incluye ejemplares en cada banda.

In [ ]:
demoras = siniestros["dias_entre_ocurrencia_reporte"].dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(demoras, bins=range(0, int(demoras.max()) + 2), color="#9B59B6", edgecolor="white")
ax.axvline(3, linestyle="--", color="#F1C40F", label="banda mid (4-7 días)")
ax.axvline(7, linestyle="--", color="#E74C3C", label="banda high (> 7 días)")
ax.set_title("Demora entre ocurrencia y reporte")
ax.set_xlabel("Días")
ax.set_ylabel("Cantidad de siniestros")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mediana: {demoras.median():.1f} días · p90: {demoras.quantile(0.9):.0f} días · máx: {int(demoras.max())} días")

## 7. Documentos: completitud e inconsistencias

`FS-08` (documentos incompletos) y `FS-11` (documentos inconsistentes) son señales clave para el reto. Verificamos que ambas tienen representación.

In [ ]:
doc_stats = pd.Series({
    "% entregados": documentos["entregado"].mean() * 100,
    "% legibles": documentos["legible"].mean() * 100,
    "% con inconsistencia": documentos["inconsistencia_detectada"].mean() * 100,
}).round(1)
display(doc_stats.to_frame("valor"))

siniestros_doc_incompleto = documentos.groupby("id_siniestro")["entregado"].agg(lambda s: not s.all()).sum()
siniestros_doc_inconsistente = documentos.groupby("id_siniestro")["inconsistencia_detectada"].any().sum()
print(f"Siniestros con al menos un documento faltante: {siniestros_doc_incompleto}")
print(f"Siniestros con al menos una inconsistencia documental: {siniestros_doc_inconsistente}")

## 8. Conclusión

El dataset sintético cumple los criterios de cobertura del reto:

- Todas las 21 reglas tienen ≥ 3 ejemplares (gate validado en `scripts/generate_dataset.py`).
- Las tres tiers están representadas (sin colapso a un sólo extremo).
- Las distribuciones de demora, monto, documentos y ramo son consistentes con los escenarios de fraude descritos en §7 del reto.
- El score aditivo separa claramente las clases simuladas (etiqueta 0 vs. 1), validando que el motor de reglas produce una señal aprendible para el clasificador supervisado (notebook 02).

Siguiente notebook: `02_modelo_fraude.ipynb` (entrenamiento LightGBM + SHAP).